# 금천구 무더위쉼터 후보시설 구축

공공시설(경로당, 도서관, 보건소, 체육관)을 모아 후보시설 목록을 만들고, 기존 무더위쉼터와 겹치는 시설을 분리한다.

산출물은 모두 `2_데이터/가공/`에 저장한다.

| 파일 | 내용 |
|---|---|
| `금천구_후보시설.csv` | 후보시설 전체. 상세주소가 같은 시설은 한 행으로 합침 |
| `금천구_후보시설_교집합.csv` | 후보시설 중 이미 무더위쉼터로 지정된 곳 |
| `금천구_후보시설_기존쉼터제외.csv` | 교집합을 뺀 후보시설 |
| `금천구_은행_체육관.csv` | 무더위쉼터 원자료의 금융기관, 체육시설 행 |

선행 노트북은 `shelter_geumcheon.ipynb`이며, 여기서 `금천구_무더위쉼터.csv`를 만든다.

## 1. 설정

좌표는 아래 순서로 찾고, 어느 것으로도 찾지 못하면 빈칸으로 둔다. 추정 좌표는 넣지 않는다.

1. 원자료나 공식 누리집에 있는 좌표 (서울시 공공도서관 현황, 서울시 생활체육포털)
2. 서울시 무더위쉼터 자료에서 시설명, 도로명주소, 지번주소 중 하나가 같은 행
3. ArcGIS World Geocoding 결과 중 주소 지점(PointAddress), 점수 100, 번지까지 일치하는 것
4. OpenStreetMap Nominatim 결과 중 도로명과 건물번호가 정확히 일치하는 것
5. OSM에 시설 이름으로 등록된 객체

지오코딩은 주소를 위도·경도로 바꾸는 작업이다. 조회 결과는 `_geocode_cache.json`에 저장하므로 다시 실행해도 외부 서버에 재요청하지 않는다.

동은 법정동(가산동, 독산동, 시흥동)으로 적는다.

In [1]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

# 이 노트북은 3_분석 폴더에서 실행한다고 가정한다.
BASE = Path.cwd().parent
RAW = BASE / "2_데이터" / "원본"
PROC = BASE / "2_데이터" / "가공"
CACHE = PROC / "_geocode_cache.json"

SIDO, GU = "서울특별시", "금천구"
DONGS = ("가산동", "독산동", "시흥동")

## 2. 주소 처리 함수

In [2]:
def norm_name(s):
    return re.sub(r"[\s()（）]", "", str(s or ""))


def road_key(s):
    """'서울특별시 금천구 시흥대로73길 70 (독산동)' -> '시흥대로73길70'"""
    s = re.sub(r"\(.*?\)", "", str(s or ""))
    s = s.replace(SIDO, "").replace(GU, "")
    s = re.sub(r"\s+", "", s)
    m = re.match(r"(.+?(?:로|길)(?:\d+[가-힣]?길)?)(\d+(?:-\d+)?)", s)
    return m.group(1) + m.group(2) if m else s


def split_road(s):
    """'시흥대로 266(203호)' -> ('시흥대로', '266')"""
    s = re.sub(r"\(.*?\)", "", str(s)).strip()
    s = re.sub(r"\s+", " ", s)
    m = re.match(r"(\S+?(?:로|길)(?:\s?\d+[가-힣]?길)?)\s*(\d+(?:-\d+)?)", s)
    if not m:
        return None, None
    return m.group(1).replace(" ", ""), m.group(2)


def dong_from_text(s):
    for d in DONGS:
        if d in str(s or ""):
            return d
    return ""


def dong_from_admin(s):
    """OSM 행정동(독산1동 등)을 법정동으로 바꾼다."""
    s = str(s or "")
    for d in DONGS:
        if s.startswith(d[:2]):
            return d
    return ""


def strip_sido_gu(s):
    s = str(s or "").replace(SIDO, "").replace("서울", "").replace(GU, "")
    return re.sub(r"\s+", " ", s).strip()


def addr_key(detail, dong):
    """도로명이면 도로명+건물번호, 지번이면 동+번지."""
    k = road_key(detail)
    if not re.match(r".+(로|길)\d", k):
        k = dong + re.match(r"[\d-]+", detail).group(0)
    return k

## 3. 지오코더

In [3]:
_cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
_H = {"User-Agent": "geumcheon-shelter-study/0.1"}


def _get(url, params, pause):
    key = url + "|" + json.dumps(params, ensure_ascii=False, sort_keys=True)
    if key not in _cache:
        r = requests.get(url, headers=_H, params=params, timeout=30)
        r.raise_for_status()
        _cache[key] = r.json()
        CACHE.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")
        time.sleep(pause)
    return _cache[key]


def arcgis(addr):
    """주소 -> (위도, 경도, 법정동). 주소 지점, 점수 100, 번지 일치만 받는다."""
    num = re.findall(r"\d+(?:-\d+)?", addr)[-1]
    js = _get("https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/findAddressCandidates",
              {"SingleLine": f"{SIDO} {GU} {addr}", "f": "json", "countryCode": "KOR", "maxLocations": 3,
               "outFields": "Addr_type,LongLabel,Nbrhd"}, 0.3)
    for c in js.get("candidates", []):
        at = c["attributes"]
        if c["score"] == 100 and at["Addr_type"] == "PointAddress" and GU in at["LongLabel"] \
                and at["LongLabel"].rstrip().endswith(" " + num):
            return c["location"]["y"], c["location"]["x"], dong_from_text(at.get("Nbrhd"))
    return None


NOMI = "https://nominatim.openstreetmap.org/search"


def nominatim_road(road_addr):
    road, num = split_road(road_addr)
    if not road:
        return None
    js = _get(NOMI, {"street": f"{num} {road}", "city": GU, "state": SIDO, "format": "jsonv2",
                     "addressdetails": 1, "limit": 10, "countrycodes": "kr"}, 1.1)
    for x in js:
        a = x.get("address", {})
        if GU not in x["display_name"]:
            continue
        if a.get("road", "").replace(" ", "") == road and a.get("house_number", "").strip(" ,") == num:
            return float(x["lat"]), float(x["lon"]), dong_from_admin(a.get("quarter") or a.get("suburb"))
    return None


def nominatim_named(osm_name, num=None):
    js = _get(NOMI, {"q": f"{osm_name}, {GU}", "format": "jsonv2", "addressdetails": 1,
                     "limit": 10, "countrycodes": "kr"}, 1.1)
    for x in js:
        a = x.get("address", {})
        if GU not in x["display_name"] or x.get("name") != osm_name:
            continue
        if num is not None and a.get("house_number", "").strip(" ,") != num:
            continue
        return float(x["lat"]), float(x["lon"]), dong_from_admin(a.get("quarter") or a.get("suburb"))
    return None


# 도로명 검색으로 안 잡히지만 OSM에 시설(또는 그 건물)이 이름으로 등록된 경우
NAMED = {
    "라이프아파트꽃마음경로당": ("라이프아파트", "60"),
    "독산배드민턴체육관": ("독산배드민턴체육관", None),
}

## 4. 무더위쉼터 대조표

서울시 무더위쉼터 원자료에서 금천구 행만 남기고, 시설명·도로명주소·지번주소로 찾을 수 있게 색인을 만든다.

In [4]:
sh_all = pd.read_csv(RAW / "서울시_무더위쉼터.csv", encoding="cp949", dtype=str)
sh = sh_all[sh_all["도로명주소"].fillna("").str.contains(GU) | sh_all["지번주소"].fillna("").str.contains(GU)].copy()
sh["nkey"] = sh["쉼터명칭"].map(norm_name)
sh["rkey"] = sh["도로명주소"].map(road_key)
sh["jkey"] = sh["지번주소"].fillna("").str.replace(f"{SIDO} {GU} ", "", regex=False).str.strip()
by_name = sh.drop_duplicates("nkey").set_index("nkey")
by_road = sh[sh["도로명주소"].notna()].drop_duplicates("rkey").set_index("rkey")
by_jibun = sh[sh["jkey"] != ""].drop_duplicates("jkey").set_index("jkey")
print("금천구 무더위쉼터:", len(sh))

금천구 무더위쉼터: 104


## 5. 좌표·동 부여 함수

In [5]:
rows = []


def add(feature, name, detail, dong="", lat=None, lon=None, src="", road=None, jibun=None):
    rec = {"시설유형": feature, "시설명": name, "동": dong, "상세주소": detail,
           "위도": lat, "경도": lon, "좌표출처": src}
    hit = None
    if lat is None:
        s = None
        if norm_name(name) in by_name.index:
            s, how = by_name.loc[norm_name(name)], "시설명 일치"
        elif road and road_key(road) in by_road.index:
            s, how = by_road.loc[road_key(road)], "도로명주소 일치"
        elif jibun and jibun in by_jibun.index:
            s, how = by_jibun.loc[jibun], "지번주소 일치"
        if s is not None:
            hit = (float(s["위도"]), float(s["경도"]), dong_from_text(s["지번주소"]),
                   f"서울시 무더위쉼터({how}: {s['쉼터명칭']})")
        if hit is None:
            g = arcgis(road or jibun)
            if g:
                hit = (*g, "ArcGIS 지오코딩(주소 지점 일치)")
        if hit is None and road:
            g = nominatim_road(road)
            if g:
                hit = (*g, "OSM Nominatim(도로명+번호 일치)")
        if hit is None and name in NAMED:
            g = nominatim_named(*NAMED[name])
            if g:
                hit = (*g, f"OSM Nominatim(명칭 일치: {NAMED[name][0]})")
    if hit:
        rec["위도"], rec["경도"], rec["좌표출처"] = hit[0], hit[1], hit[3]
        rec["동"] = rec["동"] or hit[2]
    # 동이 아직 비었으면 ArcGIS 법정동으로 채운다. 좌표는 바꾸지 않는다.
    if not rec["동"] and (road or jibun):
        g = arcgis(road or jibun)
        if g:
            rec["동"] = g[2]
    rows.append(rec)

## 6. 시설 불러오기

### 6.1 경로당

원자료는 `(서울시경로당)현황3644(25.6월말_기준)(2.csv`이다. 위쪽 4줄은 제목과 안내문이라 건너뛴다. 주소는 도로명만 있고, 청광아파트경로당 한 곳만 지번(독산동 1139)으로 적혀 있다.

In [6]:
kd = pd.read_csv(RAW / "(서울시경로당)현황3644(25.6월말_기준)(2.csv", encoding="utf-8-sig",
                 header=None, dtype=str, skiprows=4).iloc[:, :7]
kd.columns = ["연번", "시도명", "시군구명", "시설종류", "시설명", "도로명", "관할"]
kd = kd[kd["시군구명"].fillna("").str.strip() == GU]
for _, r in kd.iterrows():
    addr = str(r["도로명"]).strip()
    d = dong_from_text(addr)
    if d:
        add("경로당", r["시설명"].strip(), addr.replace(d, "").strip(), dong=d, jibun=addr)
    else:
        add("경로당", r["시설명"].strip(), addr, road=addr)
print("경로당:", len(kd))

경로당: 78


### 6.2 도서관

서울시 공공도서관 현황정보에 위도·경도가 있어 그대로 쓴다.

In [7]:
lib = pd.read_csv(RAW / "서울시_공공도서관_현황정보.csv", encoding="cp949", dtype=str)
lib = lib[lib["구명"] == GU]
for _, r in lib.iterrows():
    addr = strip_sido_gu(r["주소"])
    detail = re.sub(r"\s+", " ", re.sub(r"\((가산동|독산동|시흥동)\)", "", addr)).strip()
    add("도서관", r["도서관명"], detail, dong=dong_from_text(addr),
        lat=float(r["위도"]), lon=float(r["경도"]), src="서울시 공공도서관 현황", road=addr)
print("도서관:", len(lib))

도서관: 4


### 6.3 보건소

금천구보건소 본소와 분소 두 곳. 주소는 지번으로 받았다.

In [8]:
for name, d, detail, jibun in [
    ("금천구보건소", "시흥동", "1020 금천구 종합청사", "시흥동 1020"),
    ("금천구보건소 한내이음센터 분소", "독산동", "1088-1 금천한내이음센터 2층", "독산동 1088-1"),
    ("금천구보건소 동네방네 마을이음센터 분소", "시흥동", "961-5 동네방네 마을이음센터 2층", "시흥동 961-5"),
]:
    add("보건소", name, detail, dong=d, jibun=jibun)

### 6.4 체육관

주소는 금천구청 누리집 시설 안내(key=986, 4002, 985)에서 확인했다. 독산테니스장 좌표는 서울시 생활체육포털 시설 페이지(ft_idx=674)의 지도 좌표다.

In [9]:
SPORTS = [
    ("독산배드민턴체육관", "독산로54길 102-82", None, None, ""),
    ("금천탁구회관", "시흥대로51길 93 (시흥빗물펌프장 3층)", None, None, ""),
    ("독산테니스장", "독산로54길 288", 37.4744363273178, 126.906946187103, "서울시 생활체육포털(ft_idx=674)"),
]
for name, road, la, lo, src in SPORTS:
    add("체육관", name, road, lat=la, lon=lo, src=src, road=road)

df = pd.DataFrame(rows)
print("전체:", len(df))
df.groupby("시설유형").size()

전체: 88


시설유형
경로당    78
도서관     4
보건소     3
체육관     3
dtype: int64

## 7. 상세주소가 같은 시설 합치기

같은 아파트 단지에 경로당이 여러 개 있으면 도로명주소가 같다. 입지 분석에서는 한 지점이므로 한 행으로 합치고, 합친 수를 `시설수`에 남긴다.

In [10]:
df["_k"] = [addr_key(a, d) for a, d in zip(df["상세주소"], df["동"])]
dup = df[df.duplicated("_k", keep=False)].sort_values("_k")
for k, g in dup.groupby("_k"):
    print(k, "|", ", ".join(g["시설명"]))


def merge(g):
    first = g.iloc[0]
    return pd.Series({
        "시설유형": "/".join(dict.fromkeys(g["시설유형"])),
        "시설명": ", ".join(g["시설명"]),
        "시설수": len(g),
        "동": first["동"] or next((d for d in g["동"] if d), ""),
        "상세주소": first["상세주소"],
        "위도": next((v for v in g["위도"] if pd.notna(v)), None),
        "경도": next((v for v in g["경도"] if pd.notna(v)), None),
        "좌표출처": next((v for v in g["좌표출처"] if v), ""),
    })


cand = df.groupby("_k", sort=False).apply(merge, include_groups=False).reset_index(drop=True)
cand.insert(2, "시구동", cand["동"].map(lambda d: f"{SIDO} {GU} {d}".strip()))
cand = cand.drop(columns="동")
for c in ("위도", "경도"):
    cand[c] = cand[c].map(lambda v: "" if v is None or pd.isna(v) else round(float(v), 7))
cand.to_csv(PROC / "금천구_후보시설.csv", index=False, encoding="utf-8-sig")

print(len(df), "행 ->", len(cand), "행")
print("좌표 없음:", (cand["위도"] == "").sum(), "/ 동 없음:", (~cand["시구동"].str.endswith("동")).sum())
cand["시구동"].value_counts()

가산로99 | 두산아파트A지구경로당, 두산아파트C지역경로당
금하로793 | 벽산아파트금하경로당, 벽산아파트제1경로당, 산복경로당
금하로816 | 벽산아파트5단지경로당, 벽산타운경로당
88 행 -> 84 행
좌표 없음: 0 / 동 없음: 0


시구동
서울특별시 금천구 독산동    39
서울특별시 금천구 시흥동    38
서울특별시 금천구 가산동     7
Name: count, dtype: int64

## 8. 기존 무더위쉼터와의 교집합

`금천구_무더위쉼터.csv`와 비교해 아래 중 하나라도 맞으면 교집합으로 본다. 같은 자리에 이미 쉼터가 있으면 후보로 추가해도 커버리지가 늘지 않기 때문이다.

- 시설명 일치 (괄호 안 설명과 공백은 무시. 예: `금천구립독산도서관(1층로비)` = `금천구립독산도서관`)
- 도로명주소 일치 (도로명 + 건물번호)
- 지번주소 일치 (동 + 번지)

주소만 같은 경우는 같은 건물 안의 다른 시설일 수 있으므로 `일치기준`과 `기존쉼터명칭`을 함께 남긴다.

In [11]:
ex = pd.read_csv(PROC / "금천구_무더위쉼터.csv", dtype=str)
ex["nkey"] = ex["쉼터명칭"].map(lambda s: norm_name(re.sub(r"\(.*?\)", "", str(s))))
ex["rkey"] = ex["도로명주소"].map(lambda s: road_key(s) if pd.notna(s) else "")
ex["jkey"] = ex["지번주소"].fillna("").str.replace(f"{SIDO} {GU} ", "", regex=False).str.replace(" ", "")
print("기존 무더위쉼터:", len(ex))


def find_existing(r):
    dong = r["시구동"].split()[-1]
    names = [norm_name(re.sub(r"\(.*?\)", "", n)) for n in r["시설명"].split(", ")]
    key = addr_key(r["상세주소"], dong)
    for how, col, vals in [("시설명", "nkey", names), ("도로명주소", "rkey", [key]), ("지번주소", "jkey", [key])]:
        hit = ex[ex[col].isin(vals) & (ex[col] != "")]
        if len(hit):
            h = hit.iloc[0]
            return pd.Series({"일치기준": how, "기존쉼터명칭": h["쉼터명칭"],
                              "기존쉼터_도로명주소": h["도로명주소"], "기존쉼터_지번주소": h["지번주소"]})
    return pd.Series({"일치기준": "", "기존쉼터명칭": "", "기존쉼터_도로명주소": "", "기존쉼터_지번주소": ""})


m = pd.concat([cand, cand.apply(find_existing, axis=1)], axis=1)
inter = m[m["일치기준"] != ""]
rest = m[m["일치기준"] == ""][cand.columns]

inter.to_csv(PROC / "금천구_후보시설_교집합.csv", index=False, encoding="utf-8-sig")
rest.to_csv(PROC / "금천구_후보시설_기존쉼터제외.csv", index=False, encoding="utf-8-sig")

print("교집합:", len(inter), "/ 기존쉼터 제외:", len(rest), "/ 합계:", len(inter) + len(rest), "=", len(cand))
print(inter["일치기준"].value_counts().to_string())
pd.crosstab(m["시설유형"], m["일치기준"].replace("", "해당 없음"))

기존 무더위쉼터: 104


교집합: 46 / 기존쉼터 제외: 38 / 합계: 84 = 84
일치기준
시설명      39
도로명주소     4
지번주소      3


일치기준,도로명주소,시설명,지번주소,해당 없음
시설유형,,,,
경로당,2,37,0,35
도서관,2,2,0,0
보건소,0,0,3,0
체육관,0,0,0,3


시설명이 다른데 주소로 묶인 행이다. 같은 건물의 다른 시설인지 따로 확인이 필요하다.

In [12]:
inter[inter["일치기준"] != "시설명"][["시설명", "상세주소", "일치기준", "기존쉼터명칭"]]

,시설명,상세주소,일치기준,기존쉼터명칭
10,참새공원경로당,한내로 55,도로명주소,참새경로당
49,시흥베르빌아파트경로당,시흥대로77길 23,도로명주소,금천구시각장애인쉼터
75,금천구립금나래도서관,시흥대로73길 70,도로명주소,우리은행금천구청지점
77,금천구립시흥도서관,금하로 764 금천종합복지타운(시흥2동주민센터),도로명주소,시흥2동마을활력소(늘솔나루)
78,금천구보건소,1020 금천구 종합청사,지번주소,우리은행금천구청지점
79,금천구보건소 한내이음센터 분소,1088-1 금천한내이음센터 2층,지번주소,독산1동주민센터분소
80,금천구보건소 동네방네 마을이음센터 분소,961-5 동네방네 마을이음센터 2층,지번주소,동네방네마을이음센터(로비)


## 8.1 교집합을 뺀 기존 무더위쉼터

8절과 같은 기준(시설명, 도로명주소, 지번주소)을 쉼터 쪽에서 적용해, 후보시설 어느 하나와 일치하는 쉼터를 모두 뺀다.
교집합 파일의 `기존쉼터명칭`은 후보마다 처음 일치한 쉼터 하나만 적으므로, 한 주소에 쉼터가 여럿이면 이름 목록만으로는 덜 빠진다.

In [ ]:
cand_names = set()
for n in cand["시설명"]:
    cand_names |= {norm_name(re.sub(r"\(.*?\)", "", x)) for x in n.split(", ")}
cand_keys = {addr_key(a, d.split()[-1]) for a, d in zip(cand["상세주소"], cand["시구동"])}

hit = ex["nkey"].isin(cand_names) | ex["rkey"].isin(cand_keys) | ex["jkey"].isin(cand_keys)
ex_rest = ex.loc[~hit, [c for c in ex.columns if c not in ("nkey", "rkey", "jkey")]]
ex_rest.to_csv(PROC / "금천구_무더위쉼터_교집합제외.csv", index=False, encoding="utf-8-sig")

only_listed = set(inter["기존쉼터명칭"])
print("기존 쉼터:", len(ex), "/ 교집합으로 빠진 쉼터:", hit.sum(), "/ 남은 쉼터:", len(ex_rest))
print("교집합 파일 기존쉼터명칭(고유):", len(only_listed))
print("이름 목록에는 없지만 주소가 같아 함께 빠진 쉼터:")
ex.loc[hit & ~ex["쉼터명칭"].isin(only_listed), ["쉼터명칭", "도로명주소", "지번주소"]]

## 9. 은행·체육관

무더위쉼터 원자료에서 금천구 금융기관과 체육시설 행을 원래 열 그대로 뽑는다. 구청 누리집에서 찾은 체육관 3곳은 원자료에 없으므로 같은 열 형식으로 뒤에 붙이고 `비고`에 출처를 적는다.

In [13]:
bank = sh[sh["시설구분2"].fillna("").str.contains("금융")].copy()
gym = sh[sh["쉼터명칭"].fillna("").str.contains("체육|배드민턴|탁구|테니스")].copy()
bank.insert(0, "분류", "은행")
gym.insert(0, "분류", "체육관")
orig_cols = ["분류"] + list(sh_all.columns)
extra = cand[cand["시설유형"] == "체육관"]
sport_rows = pd.DataFrame([{
    "분류": "체육관", "시설구분1": "공공시설", "시설구분2": "체육시설", "쉼터명칭": r["시설명"],
    "도로명주소": f"{r['시구동']} {r['상세주소']}", "위도": r["위도"], "경도": r["경도"],
    "비고": "무더위쉼터 원자료에 없음. 주소는 금천구청 누리집, 좌표 출처: " + (r["좌표출처"] or "없음"),
} for _, r in extra.iterrows()], columns=orig_cols)
bg = pd.concat([bank[orig_cols], gym[orig_cols], sport_rows], ignore_index=True)
bg.to_csv(PROC / "금천구_은행_체육관.csv", index=False, encoding="utf-8-sig")
bg["분류"].value_counts()

분류
은행     11
체육관     5
Name: count, dtype: int64